<a href="https://colab.research.google.com/github/esthy13/cil-intrusion-detection/blob/main/ExperienceReplay_CIL_UNSWNB15_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/esthy13/cil-intrusion-detection

fatal: destination path 'cil-intrusion-detection' already exists and is not an empty directory.


In [ ]:
from google.colab import userdata
token = userdata.get('GIT_TOKEN')
username = userdata.get('USERNAME')
email = userdata.get('EMAIL')
!git config --global user.email {email}
!git config --global user.name {username}
repo = "cil-intrusion-detection"

In [ ]:
#importsss

import sys
import os

# Adding src folder to path
project_src_path = os.path.join(os.getcwd(), "cil-intrusion-detection", "src")
sys.path.append(project_src_path)

print("Project src path added:", project_src_path)



import torch
import torch.nn as nn
import numpy as np
import random

from torch.utils.data import DataLoader, Subset

# Project files
from model import CILModel
from task_builder import build_task, build_scenario, UpToNormalizer
from metrics import accuracy, macro_f1, compute_cm, save_confusion_matrix, average_accuracy
from utils import (
    print_task_results,
    print_scenario,
    print_strategy,
    save_training_results
)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


Project src path added: /content/cil-intrusion-detection/src
Using device: cpu


## Loading UNSW-NB15 Dataset
- Extract feature columns
- Encode class labels
- Store tensors for continual learning
- Ensure "benign" is class index 0 (required by project rules)

In [ ]:
# Correct paths
train_path = "cil-intrusion-detection/data/raw/UNSW_NB15_training-set.csv"
test_path = "cil-intrusion-detection/data/raw/UNSW_NB15_testing-set.csv"


loading the UNSW-NB15 training and testing sets.

Since the dataset contains categorical features
(e.g., proto, service, state), we apply one-hot encoding
before converting to PyTorch tensors.

We also ensure the benign class ("Normal") is class 0.

In [ ]:
#loading dataset

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Correct paths
train_path = "cil-intrusion-detection/data/raw/UNSW_NB15_training-set.csv"
test_path = "cil-intrusion-detection/data/raw/UNSW_NB15_testing-set.csv"

# Load CSV files
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Combine train + test for consistent encoding
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Separate features and labels
X = full_df.drop(columns=["label", "attack_cat"])
y = full_df["attack_cat"]

# Replace NaNs
X = X.fillna(0)

# One-hot encode categorical columns
X = pd.get_dummies(X)

print("Feature shape after encoding:", X.shape)

# ---------------------------
# Encode labels
# ---------------------------

le = LabelEncoder()
y_encoded = le.fit_transform(y)

class_names = list(le.classes_)

# Ensure benign is class 0
if "Normal" in class_names:
    benign_name = "Normal"
elif "benign" in class_names:
    benign_name = "benign"
else:
    raise ValueError("No benign/Normal class found in dataset!")

if class_names[0] != benign_name:
    print("⚠ Reordering classes so benign is class 0")
    benign_index = class_names.index(benign_name)
    class_names.insert(0, class_names.pop(benign_index))

print("Classes:", class_names)

class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

# Convert to tensors
# ---------------------------
# Ensure all features are numeric
# ---------------------------

X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(0)

# Convert explicitly to float32 numpy array
X_np = X.to_numpy(dtype=np.float32)

# Convert to tensors
X_tensor = torch.from_numpy(X_np)
y_tensor = torch.tensor([class_to_idx[c] for c in y], dtype=torch.long)

print("Final feature dimension:", X_tensor.shape[1])
print("Total samples:", len(X_tensor))
print("Tensor dtype:", X_tensor.dtype)



Train shape: (175341, 45)
Test shape: (82332, 45)
Feature shape after encoding: (257673, 197)
⚠ Reordering classes so benign is class 0
Classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
Final feature dimension: 197
Total samples: 257673
Tensor dtype: torch.float32


In [ ]:
y = full_df["attack_cat"]


In [ ]:
# ---------------------------
# Remove Worms class
# ---------------------------

mask = y != "Worms"
X = X[mask]
y = y[mask]

print("Removed Worms class.")
print("Remaining samples:", len(y))


Removed Worms class.
Remaining samples: 257499


In [ ]:
# Show remaining unique classes after removing Worms
remaining_classes = sorted(y.unique())

print("Remaining classes:")
for c in remaining_classes:
    print("-", c)

print("\nTotal number of classes:", len(remaining_classes))


Remaining classes:
- Analysis
- Backdoor
- DoS
- Exploits
- Fuzzers
- Generic
- Normal
- Reconnaissance
- Shellcode

Total number of classes: 9


In [ ]:
full_df = pd.concat([train_df, test_df], ignore_index=True)


In [ ]:
# ---------------------------
# Remove Worms class
# ---------------------------

print("Original number of samples:", len(full_df))

full_df = full_df[full_df["attack_cat"] != "Worms"]

print("After removing Worms:", len(full_df))


Original number of samples: 257673
After removing Worms: 257499


In [ ]:
X = full_df.drop(columns=["label", "attack_cat"])
y = full_df["attack_cat"]


In [ ]:
# ==========================
# REMOVE WORMS SAFELY
# ==========================

# Find Worms index
if "Worms" in class_to_idx:
    worms_idx = class_to_idx["Worms"]
    print("Worms class index:", worms_idx)
else:
    raise ValueError("Worms class not found!")

# Create mask for non-Worms samples
mask = y_tensor != worms_idx

# Filter tensors
X_tensor = X_tensor[mask]
y_tensor = y_tensor[mask]

print("Samples after removing Worms:", len(y_tensor))

# ---------------------------
# Rebuild class list WITHOUT Worms
# ---------------------------

new_class_names = [c for c in class_names if c != "Worms"]

print("New class list:")
for c in new_class_names:
    print("-", c)

print("Total classes now:", len(new_class_names))

# Rebuild class mappings
class_to_idx = {c: i for i, c in enumerate(new_class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

# Re-encode labels
y_tensor = torch.tensor(
    [class_to_idx[idx_to_class_old]
     for idx_to_class_old in
     [class_names[i] for i in y_tensor.tolist()]
    ],
    dtype=torch.long
)

print("Label remapping complete.")
print("Final dataset size:", len(y_tensor))


Worms class index: 9
Samples after removing Worms: 257499
New class list:
- Normal
- Analysis
- Backdoor
- DoS
- Exploits
- Fuzzers
- Generic
- Reconnaissance
- Shellcode
Total classes now: 9
Label remapping complete.
Final dataset size: 257499


## Custom Dataset for Continual Learning

We create a dataset class compatible with the provided
`build_task()` function.

The dataset must contain:
- x (feature tensor)
- y (label tensor)
- class_to_idx mapping
- __getitem__ and __len__ methods


In [ ]:
# ==========================
# 3️⃣ CUSTOM DATASET CLASS
# ==========================

from torch.utils.data import Dataset

class UNSWDataset(Dataset):
    def __init__(self, X_tensor, y_tensor, class_to_idx):
        self.x = X_tensor
        self.y = y_tensor
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# Create dataset instance
dataset = UNSWDataset(X_tensor, y_tensor, class_to_idx)

print("Dataset created successfully.")
print("Number of samples:", len(dataset))
print("Number of classes:", len(class_to_idx))


Dataset created successfully.
Number of samples: 257499
Number of classes: 9


## Define Class-Incremental Scenario

We define the class-incremental scenario using the provided
`build_scenario()` function.

Pattern: [2, 2, 2, 2]

This means:
- 4 tasks
- 2 new attack classes per task
- Benign (Normal) is always included


In [ ]:
# ==========================
# 5️⃣ DEFINE SCENARIO
# ==========================

# Get ordered class list (Normal must be first)
all_classes = list(class_to_idx.keys())

print("All classes:", all_classes)

# Define attack pattern (8 attacks total)
attack_pattern = [2, 2, 2, 2]

tasks, pattern = build_scenario(
    all_classes=all_classes,
    attacks_pattern=attack_pattern,
    benign_class="Normal"
)

print_scenario("ER-1", attack_pattern)

for i, task in enumerate(tasks):
    print(f"Task {i+1} classes:", task)


All classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']
=== Scenario ER-1 - [2, 2, 2, 2] ===


Task 1 classes: ['Normal', 'Analysis', 'Backdoor']
Task 2 classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits']
Task 3 classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic']
Task 4 classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']


## Initialize Model and Experience Replay Components

We initialize:

- CILModel (feature_dim = 128)
- Optimizer
- Loss function
- UpToNormalizer
- Memory buffer (size = 2000)
- Accuracy tracking matrix (for forgetting computation)


## Initialize Model and Experience Replay Components

We initialize:

- CILModel (feature_dim = 128)
- Optimizer
- Loss function
- UpToNormalizer
- Memory buffer (size = 2000)
- Accuracy tracking matrix (for forgetting computation)


In [ ]:
# ==========================
# 6️⃣ INITIALIZE MODEL & ER
# ==========================

input_dim = X_tensor.shape[1]
feature_dim = 128
total_buffer_size = 2000
learning_rate = 0.001
batch_size = 256
num_epochs = 5  # we can increase later

# Initialize model
model = CILModel(input_dim=input_dim, feature_dim=feature_dim).to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Normalizer
normalizer = UpToNormalizer()

# Experience Replay Buffer (stores indices only)
buffer_indices = []

# Accuracy matrix for forgetting computation
accuracy_matrix = []

print("Model initialized.")
print("Input dimension:", input_dim)
print("Feature dimension:", feature_dim)
print("Buffer size:", total_buffer_size)


Model initialized.
Input dimension: 197
Feature dimension: 128
Buffer size: 2000


## Define Training Function for One Task

We define a reusable training function that:

- Receives a dataset
- Applies class-weighted loss
- Trains for a fixed number of epochs
- Returns nothing (updates model in-place)


In [ ]:
# ==========================
# 7️⃣ TRAINING FUNCTION
# ==========================

def train_one_task(model, train_loader, optimizer, device, class_weights=None):
    model.train()

    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        total_loss = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, _ = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {total_loss:.4f}")


## Define Evaluation Function

This function evaluates the model on a given dataset.

It computes:
- Accuracy
- Macro-F1
- Confusion Matrix

The confusion matrix is saved to:
`results/confusion_matrices/`


In [ ]:
# ==========================
# 8️⃣ EVALUATION FUNCTION
# ==========================

def evaluate(model, dataset, seen_classes, task_id):
    model.eval()

    loader = DataLoader(dataset, batch_size=512, shuffle=False)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits, _ = model(x)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Compute metrics
    acc = accuracy(all_labels, all_preds)
    f1 = macro_f1(all_labels, all_preds)

    # Confusion matrix
    cm = compute_cm(all_labels, all_preds, seen_classes, show_plot=False)

    # Save confusion matrix
    save_path = f"cil-intrusion-detection/results/confusion_matrices/task_{task_id}.png"
    save_confusion_matrix(cm, seen_classes, save_path)

    print(f"Confusion matrix saved to: {save_path}")

    return acc, f1


## Process First Task (just checking )

Before adding Experience Replay, we process only Task 1 to verify:

- Classifier expansion
- Up-to-normalization
- Training pipeline
- Evaluation pipeline

This ensures the core CIL mechanism works correctly.


In [ ]:
# ==========================
# 9️⃣ PROCESS TASK 1 (NO ER YET)
# ==========================

task_id = 0
task_classes = tasks[task_id]

print("Processing Task 1 with classes:", task_classes)

# Build task dataset
task_dataset = build_task(dataset, task_classes)

# Update classifier for new number of classes
num_classes = len(task_classes)
model.update_classifier(num_classes)
model = model.to(device)

# ---------------------------
# Up-to-task normalization
# ---------------------------

# Extract task data for normalization
task_x = task_dataset.dataset.x[task_dataset.indices].numpy()
normalizer.update(task_x)

# Apply normalization
normalized_x = normalizer.normalize(task_x)
task_dataset.dataset.x[task_dataset.indices] = torch.tensor(
    normalized_x, dtype=torch.float32
)

# ---------------------------
# DataLoader
# ---------------------------

train_loader = DataLoader(task_dataset, batch_size=batch_size, shuffle=True)

# ---------------------------
# Train
# ---------------------------

train_one_task(model, train_loader, optimizer, device)

# ---------------------------
# Evaluate
# ---------------------------

acc, f1 = evaluate(model, task_dataset, task_classes, task_id+1)

print("Task 1 Accuracy:", acc)
print("Task 1 Macro-F1:", f1)


Processing Task 1 with classes: ['Normal', 'Analysis', 'Backdoor']
Epoch 1/5 - Loss: 262.4556
Epoch 2/5 - Loss: 258.5604
Epoch 3/5 - Loss: 258.3063
Epoch 4/5 - Loss: 258.0417
Epoch 5/5 - Loss: 257.9649
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_1.png
Task 1 Accuracy: 0.9749301063200212
Task 1 Macro-F1: 0.6518612163466001


## Define Experience Replay Buffer Update

The buffer:
- Stores ONLY dataset indices
- Has fixed total size = 2000
- Distributes memory equally across all seen classes

After each task:
- We rebalance memory per class


In [ ]:
# ==========================
# 🔟 BUFFER UPDATE FUNCTION
# ==========================

def update_buffer(buffer_indices, task_dataset, total_buffer_size, seen_classes):
    """
    buffer_indices: existing buffer
    task_dataset: dataset of current task
    total_buffer_size: 2000
    seen_classes: list of current classes
    """

    # Get all indices of current task from original dataset
    current_indices = task_dataset.indices

    # Merge old buffer with new task indices
    combined_indices = list(set(buffer_indices + list(current_indices)))

    # Compute memory per class
    num_classes = len(seen_classes)
    memory_per_class = total_buffer_size // num_classes

    new_buffer = []

    # For each class, select memory_per_class samples
    for class_id in range(num_classes):
        class_samples = [
            idx for idx in combined_indices
            if dataset.y[idx].item() == class_id
        ]

        if len(class_samples) > memory_per_class:
            class_samples = random.sample(class_samples, memory_per_class)

        new_buffer.extend(class_samples)

    print("Buffer updated.")
    print("Total buffer size:", len(new_buffer))

    return new_buffer


## Process Task 2 with Experience Replay

In this step we:

1. Expand the classifier
2. Normalize only the new task data
3. Merge buffer samples with current task samples
4. Apply class-weighted loss
5. Train the model
6. Evaluate on all seen classes
7. Update the memory buffer


In [ ]:
# ==========================
# 1️⃣1️⃣ PROCESS TASK 2 WITH ER
# ==========================

task_id = 1
task_classes = tasks[task_id]

print("\nProcessing Task 2 with classes:", task_classes)

# Build current task dataset
task_dataset = build_task(dataset, task_classes)

# Expand classifier
num_classes = len(task_classes)
model.update_classifier(num_classes)
model = model.to(device)

# ---------------------------
# Normalize ONLY new task data
# ---------------------------

task_x = task_dataset.dataset.x[task_dataset.indices].numpy()
normalizer.update(task_x)

normalized_x = normalizer.normalize(task_x)
task_dataset.dataset.x[task_dataset.indices] = torch.tensor(
    normalized_x, dtype=torch.float32
)

# ---------------------------
# Merge buffer + current task
# ---------------------------

if len(buffer_indices) > 0:
    combined_indices = buffer_indices + list(task_dataset.indices)
else:
    combined_indices = list(task_dataset.indices)

combined_dataset = Subset(dataset, combined_indices)

# ---------------------------
# Class weights (important)
# ---------------------------

labels = dataset.y[combined_indices].numpy()
class_counts = np.bincount(labels, minlength=num_classes)
class_weights = 1.0 / (class_counts + 1e-8)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class counts:", class_counts)
print("Using class weights.")

# ---------------------------
# Train
# ---------------------------

train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)

train_one_task(model, train_loader, optimizer, device, class_weights)

# ---------------------------
# Evaluate on seen classes
# ---------------------------

eval_dataset = build_task(dataset, task_classes)
acc, f1 = evaluate(model, eval_dataset, task_classes, task_id+1)

print("Task 2 Accuracy:", acc)
print("Task 2 Macro-F1:", f1)

# ---------------------------
# Update buffer
# ---------------------------

buffer_indices = update_buffer(
    buffer_indices,
    task_dataset,
    total_buffer_size,
    task_classes
)



Processing Task 2 with classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits']
Class counts: [93000  2677  2329 16353 44525]
Using class weights.
Epoch 1/5 - Loss: 797.3427
Epoch 2/5 - Loss: 789.5384
Epoch 3/5 - Loss: 787.8722
Epoch 4/5 - Loss: 787.2038
Epoch 5/5 - Loss: 786.4344
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_2.png
Task 2 Accuracy: 0.8370698119382695
Task 2 Macro-F1: 0.6228064380487426
Buffer updated.
Total buffer size: 2000


In [ ]:
# Check buffer distribution
from collections import Counter

buffer_labels = [dataset.y[idx].item() for idx in buffer_indices]
label_counts = Counter(buffer_labels)

print("Buffer distribution per class:")
for k, v in label_counts.items():
    print(f"Class {k} ({idx_to_class[k]}): {v}")

print("Total in buffer:", sum(label_counts.values()))


Buffer distribution per class:
Class 0 (Normal): 400
Class 1 (Analysis): 400
Class 2 (Backdoor): 400
Class 3 (DoS): 400
Class 4 (Exploits): 400
Total in buffer: 2000


## Initialize Accuracy Matrix for Forgetting

We maintain a matrix:

accuracy_matrix[i][j] = accuracy on task j after training task i

This allows correct computation of forgetting.


In [ ]:
# ==========================
# 1️⃣2️⃣ ACCURACY MATRIX INIT
# ==========================

num_tasks = len(tasks)

# Initialize matrix with zeros
accuracy_matrix = [
    [0 for _ in range(num_tasks)]
    for _ in range(num_tasks)
]

print("Accuracy matrix initialized with size:", num_tasks, "x", num_tasks)


Accuracy matrix initialized with size: 4 x 4


## Evaluate Model on All Seen Tasks (After Task 2)

After training Task 2, we evaluate the model on:

- Task 1
- Task 2

We store results in the accuracy matrix for proper forgetting computation.


In [ ]:
# ==========================
# 1️⃣3️⃣ EVALUATE ALL SEEN TASKS
# ==========================

current_task_index = 1  # Task 2 (0-indexed)

for prev_task_index in range(current_task_index + 1):

    prev_task_classes = tasks[prev_task_index]
    prev_dataset = build_task(dataset, prev_task_classes)

    acc, _ = evaluate(
        model,
        prev_dataset,
        prev_task_classes,
        f"{current_task_index+1}_eval_on_task_{prev_task_index+1}"
    )

    accuracy_matrix[current_task_index][prev_task_index] = acc

print("\nAccuracy matrix after Task 2:")
for row in accuracy_matrix:
    print(row)


Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_2_eval_on_task_1.png
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_2_eval_on_task_2.png

Accuracy matrix after Task 2:
[0, 0, 0, 0]
[0.9386568169295757, 0.8370698119382695, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]


In [ ]:
# Check number of optimizer parameters
total_params = sum(p.numel() for p in model.parameters())
optimizer_params = sum(p.numel() for group in optimizer.param_groups for p in group['params'])

print("Model parameters:", total_params)
print("Optimizer parameters:", optimizer_params)


Model parameters: 84229
Optimizer parameters: 83584


In [ ]:
# Reinitialize optimizer correctly
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Optimizer reinitialized.")


Optimizer reinitialized.


## Full Experience Replay Training Loop

We now implement the complete ER pipeline:

For each task:
- Expand classifier
- Reinitialize optimizer
- Normalize incrementally
- Merge replay buffer
- Train with class-weighted loss
- Evaluate on all seen tasks
- Update memory buffer
- Store accuracies

After all tasks:
- Compute average accuracy
- Compute forgetting
- Save results to JSON


In [ ]:
# ==========================
# RESET MODEL BEFORE FULL LOOP
# ==========================

model = CILModel(input_dim=input_dim, feature_dim=feature_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
normalizer = UpToNormalizer()
buffer_indices = []

print("Model reset successfully.")


Model reset successfully.


In [ ]:
# ==========================
# FULL EXPERIENCE REPLAY LOOP
# ==========================

strategy_name = "ExperienceReplay"
scenario_id = "ER-[2,2,2,2]"

print_strategy(strategy_name)
print_scenario(scenario_id, attack_pattern)

accuracy_matrix = [
    [0 for _ in range(num_tasks)]
    for _ in range(num_tasks)
]

buffer_indices = []
normalizer = UpToNormalizer()

for task_index, task_classes in enumerate(tasks):

    print(f"\nProcessing Task {task_index+1} with classes:", task_classes)

    # Build dataset for current task
    task_dataset = build_task(dataset, task_classes)

    # Expand classifier
    num_classes = len(task_classes)
    model.update_classifier(num_classes)
    model = model.to(device)

    # Reinitialize optimizer (CRITICAL)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # ---------------------------
    # Normalize new task data
    # ---------------------------
    task_x = task_dataset.dataset.x[task_dataset.indices].numpy()
    normalizer.update(task_x)

    normalized_x = normalizer.normalize(task_x)
    task_dataset.dataset.x[task_dataset.indices] = torch.tensor(
        normalized_x, dtype=torch.float32
    )

    # ---------------------------
    # Merge buffer + new task
    # ---------------------------
    if len(buffer_indices) > 0:
        combined_indices = buffer_indices + list(task_dataset.indices)
    else:
        combined_indices = list(task_dataset.indices)

    combined_dataset = Subset(dataset, combined_indices)

    # ---------------------------
    # Class weights
    # ---------------------------
    labels = dataset.y[combined_indices].numpy()
    class_counts = np.bincount(labels, minlength=num_classes)
    class_weights = 1.0 / (class_counts + 1e-8)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # ---------------------------
    # Train
    # ---------------------------
    train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)

    train_one_task(model, train_loader, optimizer, device, class_weights)

    # ---------------------------
    # Evaluate on all seen tasks
    # ---------------------------
    for prev_index in range(task_index + 1):

        prev_classes = tasks[prev_index]
        prev_dataset = build_task(dataset, prev_classes)

        acc, f1 = evaluate(
            model,
            prev_dataset,
            prev_classes,
            f"{task_index+1}_eval_on_task_{prev_index+1}"
        )

        accuracy_matrix[task_index][prev_index] = acc

        if prev_index == task_index:
            print_task_results(
                task_index+1,
                task_classes,
                prev_classes,
                acc,
                f1
            )

    # ---------------------------
    # Update buffer
    # ---------------------------
    buffer_indices = update_buffer(
        buffer_indices,
        task_dataset,
        total_buffer_size,
        task_classes
    )

# ==========================
# FINAL METRICS
# ==========================

# Average accuracy (last row mean)
final_accuracies = accuracy_matrix[num_tasks-1][:num_tasks]
avg_acc = np.mean(final_accuracies)

# Forgetting calculation (correct formula)
forgetting_values = []

for t in range(num_tasks - 1):
    max_past = max(accuracy_matrix[i][t] for i in range(num_tasks))
    final_perf = accuracy_matrix[num_tasks-1][t]
    forgetting_values.append(max_past - final_perf)

forgetting_measure = np.mean(forgetting_values)

print("\nFinal Average Accuracy:", avg_acc)
print("Final Forgetting:", forgetting_measure)

# Save JSON results
save_training_results(
    strategy_name,
    attack_pattern,
    accuracy_matrix,
    None,
    avg_acc,
    forgetting_measure,
    scenario_id,
    "cil-intrusion-detection/results/er_results.json"
)


Strategy ExperienceReplay ========


=== Scenario ER-[2,2,2,2] - [2, 2, 2, 2] ===



Processing Task 1 with classes: ['Normal', 'Analysis', 'Backdoor']
Epoch 1/5 - Loss: 229.3506
Epoch 2/5 - Loss: 180.6820
Epoch 3/5 - Loss: 173.6729
Epoch 4/5 - Loss: 170.4980
Epoch 5/5 - Loss: 169.5601
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_1_eval_on_task_1.png
   --- Task 1 ---

    New attacks: ['Normal', 'Analysis', 'Backdoor']
    Seen so far: ['Normal', 'Analysis', 'Backdoor']

    accuracy: 0.93
    macro-f1: 0.59


Buffer updated.
Total buffer size: 1998

Processing Task 2 with classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits']
Epoch 1/5 - Loss: 511.6463
Epoch 2/5 - Loss: 340.5702
Epoch 3/5 - Loss: 320.6766
Epoch 4/5 - Loss: 314.7106
Epoch 5/5 - Loss: 308.2821
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/task_2_eval_on_task_1.png
Confusion matrix saved to: cil-intrusion-detection/results/confusion_matrices/t

In [ ]:
%cd cil-intrusion-detection


/content/cil-intrusion-detection
